In [ ]:
import glob

files = glob.glob(os.path.join(TEST_DATA_DIR, '*'))
files = [f for f in files if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

if not files:
    print("Brak zdjęć.")
else:
    plt.figure(figsize=(15, 5 * ((len(files) + 2) // 3)))
    
    for i, path in enumerate(files):
        # Nazwa pliku i prawdziwa emocja
        name = os.path.basename(path)
        true_label = os.path.splitext(name)[0].split('_')[-1]
        
        # Predykcja
        img, input_data = preprocess_image(path)
        if input_data is None: continue
            
        pred = model.predict(input_data, verbose=0)
        pred_label = id_to_emotion.get(np.argmax(pred), "?")
        conf = np.max(pred)
        
        # Wykres
        plt.subplot((len(files) + 2) // 3, 3, i + 1)
        plt.imshow(img)
        
        # Podpisy: TRUE na górze, PRED na dole
        plt.title(f"TRUE: {true_label}", fontsize=14, fontweight='bold')
        col = 'green' if true_label.lower() in pred_label.lower() else 'red'
        plt.xlabel(f"PRED: {pred_label} ({conf:.0%})", color=col, fontsize=12, fontweight='bold')
        plt.xticks([]); plt.yticks([])

    plt.tight_layout()
    plt.show()

In [ ]:
IMG_SIZE = 224

def preprocess_image(path):
    try:
        img = cv2.imread(path)
        if img is None: return None, None
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        return img, np.expand_dims(img.astype('float32') / 255.0, axis=0)
    except:
        return None, None

In [ ]:
# Ładowanie modelu
try:
    model = tf.keras.models.load_model(MODEL_PATH)
    with open(MAP_PATH, 'r') as f:
        id_to_emotion = {int(k): v for k, v in json.load(f).items()}
    print("✅ Model i mapowanie załadowane")
except Exception as e:
    print(f"❌ Błąd: {e}")

In [ ]:
import os
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2

# Konfiguracja ścieżek
BASE_DIR = os.getcwd()
MODEL_PATH = os.path.join(BASE_DIR, 'saved_models', 'image_model_final.h5')
MAP_PATH = os.path.join(BASE_DIR, 'saved_models', 'image_emotion_map.json')
TEST_DATA_DIR = os.path.join(BASE_DIR, 'processed_data', 'data_test')

print(f"Model path: {MODEL_PATH}")
print(f"Map path: {MAP_PATH}")
print(f"Test data dir: {TEST_DATA_DIR}")

# Testowanie modelu na własnych zdjęciach

Ten notatnik służy do przetestowania wytrenowanego modelu na zbiorze zdjęć testowych znajdujących się w folderze `processed_data/data_test`.
Zdjęcia powinny być nazwane w formacie `imie_emocja.jpg` (np. `jakub_szczescie.jpg`).